# 02 · Import, contact selection and polarity alignment

For each participant:
1. Load the first presentation of each image, keep trials without interictal epileptiform activity,
   and apply the shaft-wise Laplacian re-reference (`preprocessing.load_first_presentation_epochs`).
2. Keep the hippocampal contacts listed in `config.HIPPOCAMPAL_CONTACTS`.
3. Replace missing samples (NaN) by 0. Contact-trials without data therefore become flat and are
   excluded in `04_statistics_and_figures.ipynb`.
4. Invert the polarity of the contacts listed in `config.POLARITY_FLIPS`, so that the evoked response
   has the same orientation across contacts. The flips were chosen by visual inspection of each
   contact's trial-averaged response (inspection plots before and after flipping are saved below).

Output: `derivatives/epochs_hippocampus_aligned.pickle` (dict: participant → `mne.EpochsArray`).

In [ ]:
import pickle

import matplotlib.pyplot as plt
import numpy as np

import config
import preprocessing

config.DERIVATIVES_DIR.mkdir(parents=True, exist_ok=True)
inspection_dir = config.FIGURES_DIR / "polarity_inspection"
inspection_dir.mkdir(parents=True, exist_ok=True)

## Load and select hippocampal contacts

In [ ]:
hippocampal_epochs = {}
for patient_id in config.PATIENTS:
    epochs = preprocessing.load_first_presentation_epochs(
        patient_id, config.SEEG_DATA_ROOT, config.MEMORABILITY_FILE, task=config.TASK)

    wanted = config.HIPPOCAMPAL_CONTACTS[patient_id]
    picks = [ch for ch in epochs.ch_names if ch in wanted]  # keep recording order
    not_recorded = [ch for ch in wanted if ch not in epochs.ch_names]
    if not_recorded:
        print(f"{patient_id}: listed contacts not present in the recording: {not_recorded}")

    hippocampal_epochs[patient_id] = epochs.pick(picks)
    del epochs

n_contacts = sum(len(ep.ch_names) for ep in hippocampal_epochs.values())
print(f"{n_contacts} hippocampal contacts in {len(hippocampal_epochs)} participants")

## Polarity inspection and alignment

In [ ]:
def plot_contact_averages(epochs, title, path, n_cols=4):
    """Trial-averaged response of each contact (used to decide and check polarity flips)."""
    evoked = epochs.average()
    n_rows = int(np.ceil(len(epochs.ch_names) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 6, n_rows * 4), sharex=True, sharey=True,
                             squeeze=False)
    axes = axes.ravel()
    for ax, (ch_name, trace) in zip(axes, zip(epochs.ch_names, evoked.data)):
        ax.plot(evoked.times, trace, color="black")
        ax.set_title(ch_name, fontsize=12, fontweight="bold")
        for t in (0, 0.4, 3.0):  # image onset, ~400 ms, image offset
            ax.axvline(t, color="gray", linestyle=":", alpha=0.7)
        ax.axhline(0, color="gray", linestyle=":", alpha=0.7)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes[len(epochs.ch_names):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=16)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def invert_polarity(x):
    return -x


aligned_epochs = {}
for patient_id, epochs in hippocampal_epochs.items():
    epochs = epochs.copy()
    epochs._data = np.nan_to_num(epochs._data)  # missing samples -> 0
    plot_contact_averages(epochs, f"{patient_id} - before polarity alignment",
                          inspection_dir / f"{patient_id}_pre_flip.png")

    flips = [ch for ch in config.POLARITY_FLIPS.get(patient_id, []) if ch in epochs.ch_names]
    if flips:
        print(f"Flipping polarity for {patient_id}: {flips}")
        epochs.apply_function(invert_polarity, picks=flips)

    plot_contact_averages(epochs, f"{patient_id} - after polarity alignment",
                          inspection_dir / f"{patient_id}_post_flip.png")
    aligned_epochs[patient_id] = epochs

n_flipped = sum(len([c for c in config.POLARITY_FLIPS.get(p, []) if c in ep.ch_names])
                for p, ep in aligned_epochs.items())
print(f"Polarity inverted for {n_flipped} of {n_contacts} contacts")

In [ ]:
with open(config.ALIGNED_EPOCHS_FILE, "wb") as f:
    pickle.dump(aligned_epochs, f)
print(f"Saved {config.ALIGNED_EPOCHS_FILE}")